# LexChain summarization comparison

Three approaches on OHR-Bench Law docs (deterministic sample of 13):
1. **TextRank** (extractive, sumy) 2. **LexRank** (extractive, sumy) 3. **Llama 3.1 70B via NVIDIA NIM** (abstractive — our system).

**Two evaluation layers:** automated *proxy* metrics (ROUGE / BERTScore vs source — flagged, extractive inflates ROUGE) and a **blind human-evaluation scaffold** (the primary basis for the conclusion).

Resumable: every summary is checkpointed to Drive (`MyDrive/lexchain_bench/summarization/`), so a disconnect loses nothing.

**Setup:** Runtime → T4 GPU (for BERTScore); add your NVIDIA key via the **Secrets** panel (🔑) as `NVIDIA_API_KEY` (free at build.nvidia.com).

In [ ]:
# Cell 1 — mount Drive, clone/pull repo, install deps (torch stays Colab's build)
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/lexchain-chunk-embed-bench'):
    !git clone https://github.com/MarvinPescos/lexchain-chunk-embed-bench.git /content/lexchain-chunk-embed-bench
%cd /content/lexchain-chunk-embed-bench
!git pull
!pip install -q -r summarization/requirements-summ.txt
import nltk
for pkg in ('punkt', 'punkt_tab'):
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print('nltk', pkg, e)
print('deps ready')

In [ ]:
# Cell 2 — NVIDIA NIM key from Colab Secrets, and download OHR-Bench Law data
from google.colab import userdata
os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_API_KEY')  # LLM_PROVIDER defaults to nim
# Fallback: set LLM_PROVIDER=groq and GROQ_API_KEY instead if NIM is impractical.
!python download_data.py

In [ ]:
# Cell 3 — smoke test: 2 docs x 3 approaches, then project the full 13-doc run
%cd /content/lexchain-chunk-embed-bench/summarization
!python summarize.py --limit-docs 2

import json, glob
from pathlib import Path
cache = Path('/content/drive/MyDrive/lexchain_bench/summarization/summaries')
recs = [json.loads(p.read_text()) for p in cache.glob('*.json')]
llm = [r for r in recs if r['approach'] == 'llm']
if llm:
    per_doc_llm = sum(r['latency_s'] for r in llm) / len(llm)
    print(f"\nLLM latency/doc ~ {per_doc_llm:.1f}s -> full 13 docs ~ {per_doc_llm*13/60:.1f} min")
    print('(extractive approaches are near-instant; add ~2-3 min for BERTScore in Cell 5)')
for r in sorted(recs, key=lambda x: (x['doc'], x['approach']))[:6]:
    print(f"\n--- {r['doc']} / {r['approach']} ({r['summary_words']}w) ---\n{r['summary'][:400]}")

**STOP — review the smoke summaries and the projected time above before the full run.** The full run is resumable, so a disconnect during Cell 4 just means rerunning it.

In [ ]:
# Cell 4 — full run: 13 docs x 3 approaches (resumable; cached summaries are skipped)
!python summarize.py

In [ ]:
# Cell 5 — Layer 1: automated PROXY metrics (ROUGE + BERTScore vs source)
# Prints the proxy-metric disclaimer; extractive ROUGE is inflated by design.
!python evaluate_summ.py

In [ ]:
# Cell 6 — Layer 2: build the blind human-evaluation scaffold
# blind_eval.csv (approach hidden) + unblinding_key.csv (keep separate!)
!python build_blind_eval.py
import pandas as pd
display(pd.read_csv('/content/drive/MyDrive/lexchain_bench/summarization/blind_eval.csv').head(6))
print('Give raters blind_eval.csv (they open each doc_reference to judge). Do NOT share unblinding_key.csv.')

In [ ]:
# Cell 7 — final table (run now for the proxy-only table; re-run after raters fill
# blind_eval.csv to fold in the human coverage/accuracy/fluency columns)
!python aggregate_human.py